# SupportIQ — Stage 2: Unified Data Pipeline & SFT Dataset Preparation

> **Goal:** Run the unified data pipeline that transforms raw customer support data into fine-tuning datasets.
> **Two Critical Invariants:**
> 1. **Zero Data Leakage:** Near-duplicate instruction templates are grouped by `cluster_id` so that test examples evaluate true generalization, not memorization.
> 2. **Structured SFT Chat Format:** Formats data into standard conversation format for Qwen fine-tuning, where the assistant returns structured JSON (`{"category", "intent", "response"}`).


### 1. Setup & Environment
Import the unified pipeline from `supportiq.data`.


In [1]:
import json
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from supportiq.data.pipeline import run_full_pipeline

print("Pipeline modules loaded successfully.")


Pipeline modules loaded successfully.


### 2. Execute the Full Data Pipeline
This single call performs:
1. **Normalization:** Standardizes Unicode NFC and cleans whitespace.
2. **Clustering:** Assigns `cluster_id` to each row using 3-word MinHash LSH.
3. **Grouped Split:** Allocates 80% Train, 10% Val, and 10% Test by cluster.
4. **Leakage Audit:** Verifies mathematically that 0 clusters overlap.
5. **Artifact Export:** Writes Parquet splits and standard SFT conversational JSONL files.


In [2]:
output_directory = project_root / "data/processed"

print("Starting end-to-end data pipeline...")
audit_results = run_full_pipeline(
    raw_data_dir=project_root / "data/raw",
    output_dir=output_directory,
    seed=42,
)

print("\nPipeline execution complete!")


Starting end-to-end data pipeline...
{"timestamp": "2026-09-21T23:07:47.903380+00:00", "level": "INFO", "name": "supportiq.data.pipeline", "message": "Computing MinHash fingerprints for 26872 rows (threshold=0.80)...", "module": "pipeline", "line": 53}


{"timestamp": "2026-09-21T23:07:59.309559+00:00", "level": "INFO", "name": "supportiq.data.pipeline", "message": "Assigned 23297 unique clusters across 26872 records.", "module": "pipeline", "line": 84}


{"timestamp": "2026-09-21T23:07:59.367375+00:00", "level": "INFO", "name": "supportiq.data.pipeline", "message": "Split complete: Train=21497 (80.0%), Val=2687 (10.0%), Test=2688 (10.0%)", "module": "pipeline", "line": 151}


{"timestamp": "2026-09-21T23:08:00.714528+00:00", "level": "INFO", "name": "supportiq.data.pipeline", "message": "Exported 21497 SFT conversational records to /home/dvinix/Projects/supportiq/data/processed/train.jsonl", "module": "pipeline", "line": 233}


{"timestamp": "2026-09-21T23:08:00.876741+00:00", "level": "INFO", "name": "supportiq.data.pipeline", "message": "Exported 2687 SFT conversational records to /home/dvinix/Projects/supportiq/data/processed/val.jsonl", "module": "pipeline", "line": 233}


{"timestamp": "2026-09-21T23:08:01.035915+00:00", "level": "INFO", "name": "supportiq.data.pipeline", "message": "Exported 2688 SFT conversational records to /home/dvinix/Projects/supportiq/data/processed/test.jsonl", "module": "pipeline", "line": 233}


{"timestamp": "2026-09-21T23:08:01.073837+00:00", "level": "INFO", "name": "supportiq.data.pipeline", "message": "Exported 500 SFT conversational records to /home/dvinix/Projects/supportiq/data/processed/test_eval_500.jsonl", "module": "pipeline", "line": 233}


{"timestamp": "2026-09-21T23:08:01.074915+00:00", "level": "INFO", "name": "supportiq.data.pipeline", "message": "Full pipeline completed successfully! All artifacts saved to /home/dvinix/Projects/supportiq/data/processed", "module": "pipeline", "line": 273}



Pipeline execution complete!


### 3. Leakage Audit & Split Statistics
Confirm that our split preserves statistical validity and prevents memorization leakage.


In [3]:
print("=== SPLIT & LEAKAGE AUDIT ===")
print(f"Train Set:           {audit_results['train_rows']:,} rows ({audit_results['train_clusters']:,} unique clusters)")
print(f"Validation Set:      {audit_results['val_rows']:,} rows ({audit_results['val_clusters']:,} unique clusters)")
print(f"Test Set:            {audit_results['test_rows']:,} rows ({audit_results['test_clusters']:,} unique clusters)")
print(f"Cluster Overlap:     {audit_results['leakage_overlap']} (Mathematically guaranteed zero leakage)")
print(f"Leakage-Free Status: {'PASS (100% Leakage-Free)' if audit_results['is_leakage_free'] else 'FAIL'}")


=== SPLIT & LEAKAGE AUDIT ===
Train Set:           21,497 rows (18,598 unique clusters)
Validation Set:      2,687 rows (2,389 unique clusters)
Test Set:            2,688 rows (2,310 unique clusters)
Cluster Overlap:     0 (Mathematically guaranteed zero leakage)
Leakage-Free Status: PASS (100% Leakage-Free)


### 4. Inspect Formatted SFT Training Example
Inspect a sample row from `train.jsonl` to verify the conversational schema and structured JSON assistant payload.


In [4]:
train_jsonl_path = output_directory / "train.jsonl"

with open(train_jsonl_path, encoding="utf-8") as f:
    sample_line = f.readline()

sample_record = json.loads(sample_line)
print("=== SFT CHAT FORMAT EXAMPLE ===")
for msg in sample_record["messages"]:
    role = msg["role"].upper()
    content = msg["content"]
    if role == "ASSISTANT":
        # Pretty-print assistant JSON payload
        parsed_payload = json.loads(content)
        print(f"[{role}]:")
        print(json.dumps(parsed_payload, indent=2))
    else:
        print(f"[{role}]: {content}")
        print()


=== SFT CHAT FORMAT EXAMPLE ===
[SYSTEM]: You are SupportIQ, an expert customer support triage assistant. Classify the customer request into category and intent, and provide a helpful, polite response.

[USER]: question about cancelling order {{Order Number}}

[ASSISTANT]:
{
  "category": "ORDER",
  "intent": "cancel_order",
  "response": "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."
}


### 5. Summary of Processed Artifacts
Verify that all required data artifacts exist in `data/processed/`.


In [5]:
print("=== GENERATED ARTIFACTS IN data/processed/ ===")
for p in sorted(output_directory.iterdir()):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"  - {p.name:<25} ({size_mb:.2f} MB)")

print("\n=> STAGE 2 IS COMPLETE! Data is ready for Stage 3 (Baseline) & Stage 4 (Fine-Tuning)!")


=== GENERATED ARTIFACTS IN data/processed/ ===
  - .gitkeep                  (0.00 MB)
  - test.jsonl                (2.70 MB)
  - test.parquet              (0.35 MB)
  - test_eval_500.jsonl       (0.50 MB)
  - test_eval_500.parquet     (0.08 MB)
  - train.jsonl               (21.37 MB)
  - train.parquet             (2.49 MB)
  - val.jsonl                 (2.67 MB)
  - val.parquet               (0.34 MB)

=> STAGE 2 IS COMPLETE! Data is ready for Stage 3 (Baseline) & Stage 4 (Fine-Tuning)!
